In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

approvals = spark.table("scottish_housing_ws.silver.boe_mortgage_approvals")
hpi = spark.table("scottish_housing_ws.silver.uk_hpi")

In [0]:
hpi_scotland = (
    hpi
    .filter(F.col("area_code") == "S92000003")
    .select("report_date", "year_month", "region_name", "area_code", "average_price", "sales_volume")
)

In [0]:
approvals_window = Window.orderBy("year_month")

approvals_lagged = (
    approvals
    .withColumn(
        "approvals_lagged_3m",
        F.lag("mortgage_approvals_count", -3).over(approvals_window)
    )
    .select("year_month", "mortgage_approvals_count", "approvals_lagged_3m")
)

In [0]:
gold = (
    hpi_scotland
    .join(approvals_lagged, on="year_month", how="inner")
    .select(
        "report_date",
        "year_month",
        "region_name",
        "area_code",
        "average_price",
        "sales_volume",
        "mortgage_approvals_count",
        "approvals_lagged_3m"
    )
    .orderBy("report_date")
)

In [0]:
print(f"Row count: {gold.count()}")
gold.orderBy("report_date").show(5)
gold.orderBy(F.col("report_date").desc()).show(5)

In [0]:
(
    gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.gold.mortgage_approvals_vs_transaction_volume")
)

In [0]:
(
    gold.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv("abfss://data@scottishhousingdl.dfs.core.windows.net/exports/gold_mortgage_approvals_vs_transaction_volume/")
)

print("Gold 04 complete.")